# ADK — Simple Agent

**Framework:** Google Agent Development Kit (ADK)  
**Level:** 01 — Simple  
**Model:** `gemini-2.0-flash` (Gemini API key, no GCP needed)

### What You Will Learn
- How to define tools as plain Python functions with type-annotated docstrings
- How to create an ADK `Agent` with a system instruction and a tool list
- How the **ReAct loop** works: the model reads the query, decides to call a tool, observes the result, then responds
- How to run the agent locally with `InMemoryRunner`

## Concept: The ReAct Loop

```
User Query
    │
    ▼
┌──────────────────────────────┐
│  LLM  (gemini-2.0-flash)     │
│                              │
│  Thought: I need weather     │
│  Action:  get_weather(city)  │◄── Tool call decision
└──────────────┬───────────────┘
               │
               ▼
        [get_weather]          ← Tool executes
               │
               ▼
    Observation: {temp: 25}    ← Result fed back to LLM
               │
               ▼
┌──────────────────────────────┐
│  LLM synthesizes final answer│
└──────────────────────────────┘
               │
               ▼
         Final Response
```

**Key insight:** The LLM never 'runs' the tool directly. It outputs a *structured tool call request*, ADK intercepts it, executes the Python function, and feeds the result back as an observation. The LLM then uses that observation to generate the final answer.

At the **Simple** level:
- 1–2 tools, mock data (no API keys for tools)
- Single-turn: one query → one response
- No memory between calls

## Setup

In [ ]:
import asyncio
import os
from dotenv import load_dotenv

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()  # reads GOOGLE_API_KEY from .env

# Verify the key is loaded (never print the value)
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set — check your .env file"
print("✓ GOOGLE_API_KEY loaded")

## Tool Definitions

ADK discovers tools automatically from **Python function signatures + docstrings**.  
The docstring becomes the tool description the LLM reads when deciding whether to call it.  
The `Args:` section documents parameters — ADK generates the JSON schema from type hints.

> **Mock tools:** These return hardcoded data so you can run this notebook without any external API keys.

In [ ]:
def get_weather(city: str) -> dict:
    """Return a mock weather report for the given city.

    Args:
        city: The name of the city to get weather for.

    Returns:
        A dictionary with weather details including condition, temperature, and humidity.
    """
    mock_data = {
        "london":    {"condition": "Cloudy", "temp_c": 12, "humidity": 78},
        "new york":  {"condition": "Sunny",  "temp_c": 22, "humidity": 45},
        "bangalore": {"condition": "Rainy",  "temp_c": 25, "humidity": 85},
        "tokyo":     {"condition": "Clear",  "temp_c": 18, "humidity": 60},
    }
    city_key = city.lower()
    if city_key in mock_data:
        data = mock_data[city_key]
        return {
            "city": city,
            "condition": data["condition"],
            "temperature_celsius": data["temp_c"],
            "humidity_percent": data["humidity"],
        }
    return {"error": f"No data for '{city}'. Try: London, New York, Bangalore, Tokyo."}


def get_time(city: str) -> dict:
    """Return the current local time for the given city.

    Args:
        city: The name of the city to get the local time for.

    Returns:
        A dictionary with the city name and its local time string.
    """
    mock_times = {
        "london":    "14:30 GMT",
        "new york":  "09:30 EST",
        "bangalore": "20:00 IST",
        "tokyo":     "22:30 JST",
    }
    city_key = city.lower()
    if city_key in mock_times:
        return {"city": city, "local_time": mock_times[city_key]}
    return {"error": f"No time data for '{city}'. Try: London, New York, Bangalore, Tokyo."}


# Quick sanity check
print("get_weather('Bangalore'):", get_weather("Bangalore"))
print("get_time('Tokyo'):       ", get_time("Tokyo"))

## Agent Definition

An ADK `Agent` bundles together:
- **`model`** — which LLM to use
- **`instruction`** — the system prompt (shapes personality + behavior)
- **`tools`** — list of Python functions the model can call
- **`name` / `description`** — used in multi-agent setups (optional here)

In [ ]:
agent = Agent(
    name="weather_assistant",
    model="gemini-2.0-flash",
    description="A helpful assistant that answers weather and time questions.",
    instruction="""You are a helpful travel assistant.
When asked about weather or local time in a city, use the available tools.
Always be concise and friendly in your responses.""",
    tools=[get_weather, get_time],
)

print(f"Agent '{agent.name}' created with {len(agent.tools)} tools")

## Run the Agent

ADK uses an async runner model. `InMemoryRunner` handles everything locally:
- Session state (in memory)
- Tool execution
- Streaming events from the model

We listen for `is_final_response()` events to extract the agent's answer.

In [ ]:
async def run_agent(query: str) -> str:
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="weather_app",
        user_id="user_01",
    )
    runner = InMemoryRunner(agent=agent, session_service=session_service)

    response_text = ""
    async for event in runner.run_async(
        user_id=session.user_id,
        session_id=session.id,
        new_message=genai_types.Content(
            role="user",
            parts=[genai_types.Part(text=query)],
        ),
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text


# Run query 1 — single tool call
query1 = "What's the weather like in Bangalore right now?"
print(f"User: {query1}")
response1 = await run_agent(query1)
print(f"Agent: {response1}")

In [ ]:
# Run query 2 — two tool calls in one query
query2 = "What time is it in Tokyo, and is it a good day to go outside?"
print(f"User: {query2}")
response2 = await run_agent(query2)
print(f"Agent: {response2}")

In [ ]:
# Try your own query
your_query = "What's the weather in London and what time is it there?"
print(f"User: {your_query}")
response3 = await run_agent(your_query)
print(f"Agent: {response3}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Tools are plain Python functions | No special ADK class needed — just type hints + docstrings |
| `instruction=` is the system prompt | Controls agent personality and behavior |
| `InMemoryRunner` handles the loop | ADK manages tool call → execution → observation automatically |
| Single-turn, no memory | Each `run_agent()` call is independent — next level adds memory |
| `is_final_response()` | Event-based streaming — you can also capture intermediate tool events |

### What Changes at the Next Level (02-Intermediate)
- Adds **conversation memory** (the agent remembers previous turns in the same session)
- Adds a **third tool** (more complex decision-making)
- Returns **structured output** (Pydantic model instead of free text)

### Things to Try
- Add a new city to the mock data and ask about it
- Add a third tool (e.g., `get_forecast(city, days)`) and ask for a multi-day forecast
- Change the `instruction` to be more formal or more casual and observe the tone change
- Ask about a city not in the mock data and observe how the agent handles the error

### Next Notebook
[02-Intermediate →](../02-intermediate/agent.ipynb)